In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison

from plotly.io import to_html
from IPython.display import display, HTML
import cvxpy as cp

from modules.optimization_algorithms.NashProductOptAlgo import NashProductOptAlgo
from modules.optimization_algorithms.XNashProductOptAlgo import XNashProductOptAlgo
from modules.optimization_algorithms.EqualWaterfillingOptAlgo import EqualWaterfillingOptAlgo

In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

In [ ]:
# # PARMS
# changeable
org_id = 1
start_time = datetime(2025, 6, 21)
end_time = datetime(2025, 6, 22)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

basepath=f"../../local_data/org_{org_id}/EqualWaterfillingOptAlgo/"

timestamp = "2026-01-15T12:09"
opt_energy_filename = f"simulated_energy_{timestamp}.csv"
tfs_filename = f"tf_schedule_{timestamp}.csv"

In [ ]:
raw_eeg = pd.read_csv(f"{basepath}/{opt_energy_filename}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

### Apply pf schedule to energy data

In [ ]:
applied_pfs = raw_eeg

applied_pfs["comm_cov_delta"] = applied_pfs["opt_comm_cov"] - applied_pfs["comm_cov"]
applied_pfs["cons_gen_delta"] = applied_pfs["opt_cons_gen"] - applied_pfs["cons_gen"]

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_timestamp_from_tf_opt = np.random.choice(applied_pfs.time)
print(
    "Random timestamp from time where pf opt got applied:", random_timestamp_from_tf_opt
)

# 2) Filter dt by this value
single_time_filtered = applied_pfs[applied_pfs["time"] == random_timestamp_from_tf_opt]

check_full_calc_via_single_timestamp = (
    single_time_filtered
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
print(check_full_calc_via_single_timestamp.sum(numeric_only=True))
check_full_calc_via_single_timestamp.sort_values(by="sum_comm_cov_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_comm_cov", "sum_opt_comm_cov")